## Prompt

In [3]:

prompt = """You are an experienced Computer Science professor and MSc/PhD supervisor with 20+ years of experience. 
Your task is to carefully analyze a thesis abstract and extract the key **resource requirements** needed to carry out the described research.

Be strict and conservative. Do NOT hallucinate or invent resources that are not supported by the text. If the abstract is from a non-Computer Science field, still extract meaningfully but do not force CS-style tools.
Try to be as verbatim as possible in extracting the requirements, and do not infer beyond what is supported by the text. If a requirement is implied but not explicitly stated, still include it, but do not add any requirements that are not supported by the text.

---

### Example

**Abstract:**
"This thesis is an examination of the national narratives contained in three exhibits in The Musem of New Zealand, Te Papa Tongarewa. It examines the existence of the state and the nation, and their involvement in museum development, and applies this theory, and selected theories of Roland Barthes, Sergei Eisenstein, and Walter Benjamin, to the subsequent analysis. Broadly, the position taken is that museums are one of a number of institutions that perpetuate national narratives in order to bind nations together and discourage anti-state sentiment, and this position is validated in the analysis of three long-term Te Papa exhibits, Exhibiting Ourselves, Parade, and Golden Days."

**Expected Output:**
```json
{{
  "overall_suitable_degree": "MSc",
  "requirements": [
    {{
      "category": "method_technique",
      "name": "National narrative analysis",
      "description": "Examination of national narratives in museum exhibits",
      "estimated_difficulty": "intermediate",
      "suitable_degree": "MSc"
    }},
    {{
      "category": "method_technique",
      "name": "Application of cultural theory",
      "description": "Application of theories from Roland Barthes, Sergei Eisenstein, and Walter Benjamin",
      "estimated_difficulty": "advanced",
      "suitable_degree": "PhD"
    }},
    {{
      "category": "data",
      "name": "Te Papa Tongarewa exhibits",
      "description": "Analysis of three long-term exhibits: Exhibiting Ourselves, Parade, and Golden Days",
      "estimated_difficulty": "intermediate",
      "suitable_degree": "MSc"
    }},
    {{
      "category": "method_technique",
      "name": "Institutional analysis",
      "description": "Analysis of the role of the state and nation in museum development",
      "estimated_difficulty": "intermediate",
      "suitable_degree": "MSc"
    }}
  ]
}}

---

Now apply the same analysis to the following abstract:

[THESIS ABSTRACT]
{abstract}
[/THESIS ABSTRACT]

Focus on what would actually be required in terms of:
- Datasets / data sources
- Methods, algorithms, or theoretical techniques
- Tools, libraries, or frameworks
- Compute / hardware resources
- Human effort / skills
- Any other critical resources

### Field generation rules by category

**For `method_technique` and `tool_library`:**
Generate an `search_query` field.
- Use technical terms, acronyms, and key context words from the abstract.
- Use short, precise phrases of 3–4 words to capture the core of the technical or methodological requirement.
- Do not use author names in these queries; focus on the method or tool itself.

**For `data`:**
generate a `search_query` field that is
intended to locate THIS specific dataset or corpus if it exists
  publicly. Use the most identifying information available in the abstract —
  typically the dataset name, the citing author if named, or the institution.
  Template: [dataset_name or corpus_identifier] [domain] [language]


### Output Format
Return a valid JSON object with exactly this structure:

```json
{{
  "overall_suitable_degree": "BSc" | "MSc" | "PhD" | "MSc-to-PhD",
  "requirements": [
    {{
      "category": "data" | "method_technique" | "tool_library" | "compute" | "human_effort" | "other",
      "name": "Short clear name of the resource",
      "description": "One sentence explaining how it is used in the work",
      "estimated_difficulty": "basic" | "intermediate" | "advanced",
      "search_query": "Optimized search query for this requirement"
    }}
  ]
}}
```

Return only valid JSON. Do not include any explanation, markdown fencing, or preamble outside the JSON object.
    """

## Boilerplate

In [4]:
from pydantic import BaseModel
from typing import List
from enum import Enum

class DegreeLevel(str, Enum):
    BSc = "BSc"
    MSc = "MSc"
    PhD = "PhD"
    MSc_to_PhD = "MSc-to-PhD"

class Category(str, Enum):
    data = "data"
    method_technique = "method_technique"
    tool_library = "tool_library"
    compute = "compute"
    human_effort = "human_effort"
    other = "other"

class Difficulty(str, Enum):
    basic = "basic"
    intermediate = "intermediate"
    advanced = "advanced"

class Requirement(BaseModel):
    category: Category
    name: str
    description: str
    estimated_difficulty: Difficulty
    search_query: str

class AnalysisResult(BaseModel):
    overall_suitable_degree: DegreeLevel
    requirements: List[Requirement]

## Code

In [5]:
from tools import get_model
from dotenv import load_dotenv
import os
import json

load_dotenv(dotenv_path=".env")

llm = get_model()

In [6]:
import pandas as pd

df = pd.read_excel("/Users/ariq/Public/Data/thesis_feasibility/New Dataset.xlsx", sheet_name="filtered")

In [79]:
example_abstract = df[df["Academic Level"]=="MA"][["title","Abstract","Published Year","Academic Level"]][67:68]
title, abstract, published_year, academic_level = example_abstract["title"].values[0], example_abstract["Abstract"].values[0], example_abstract["Published Year"].values[0], example_abstract["Academic Level"].values[0]
print(abstract)

Social media has become one of the prime communication strategies of political campaigns. 
User Generated Content (UGC) and candidate posts reflect both electoral campaign strategy 
and message as well as opinions, following and preference by the electorate. An in-depth 
analysis of the UGC  provides precious information about the political issues which concern 
the electorate. In this study, an attempt to show the correlation of sentiment of  UGC with the 
electoral results will be made. Using various Key Performance Indicators (KPIs) including in
depth content analysis, Facebook Page Performance Index (PPI), volume of content, social 
media users’ engagement rates and other correlations portraying the electoral campaign 
performance of Members of the European Parliament (MEP) candidates for the 2019 election 
will be attempted. The findings of this study indicates that organic social media followers are 
correlated to the number of votes garnered in the actual election. Sentiment pol

In [ ]:
example_abstract = df[df["Academic Level"]=="PhD"][["title","Abstract","Published Year","Academic Level"]]
# Set display options
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Display the dataframe
example_abstract.reset_index().rename(columns={'index': 'row_number'}) 
# 51/67

### Extraction

In [51]:
result = llm.invoke(prompt.format(abstract=abstract))
raw = json.loads(result.content)
parsed = AnalysisResult.model_validate(raw)
print(result.content)

{
  "overall_suitable_degree": "PhD",
  "requirements": [
    {
      "category": "data",
      "name": "German financial texts",
      "description": "Analysis of German-speaking finance-related texts for sentiment analysis.",
      "estimated_difficulty": "intermediate",
      "search_query": "German financial texts finance German"
    },
    {
      "category": "data",
      "name": "German finance-specific dictionary",
      "description": "The first finance-specific German dictionary developed by Bannier et al. (2019) for sentiment analysis.",
      "estimated_difficulty": "intermediate",
      "search_query": "German finance-specific dictionary Bannier 2019"
    },
    {
      "category": "data",
      "name": "Annual reports of German-speaking banks",
      "description": "Sentiment analysis of German-speaking annual reports to predict financial performance metrics.",
      "estimated_difficulty": "intermediate",
      "search_query": "German-speaking annual reports finance"
   

In [53]:
requirements_df = pd.DataFrame([
    {
        "category": req.category,
        "name": req.name,
        "description": req.description,
        "arxiv_query": req.search_query
    }
    for req in parsed.requirements
])

requirements_df

,category,name,description,arxiv_query
0,Category.data,German financial texts,Analysis of German-speaking finance-related texts for sentiment analysis.,German financial texts finance German
1,Category.data,German finance-specific dictionary,The first finance-specific German dictionary developed by Bannier et al. (2019) for sentiment analysis.,German finance-specific dictionary Bannier 2019
2,Category.data,Annual reports of German-speaking banks,Sentiment analysis of German-speaking annual reports to predict financial performance metrics.,German-speaking annual reports finance
3,Category.method_technique,Sentiment analysis,Enhancement of sentiment analysis accuracy and reliability for German financial texts.,Sentiment analysis finance German texts
4,Category.method_technique,Neural network training,Training of neural networks based on Word2vec to create a comprehensive dictionary.,Neural network training Word2vec
5,Category.method_technique,Dictionary expansion,Expansion of the finance-specific dictionary with additional words and synonyms.,Dictionary expansion finance German
6,Category.tool_library,Word2vec,Used for training neural networks to create a comprehensive dictionary.,Word2vec text analysis
7,Category.human_effort,Textual analysis skills,Skills in textual analysis and sentiment analysis for finance-related texts.,Textual analysis sentiment analysis finance
8,Category.other,Manual classification,Manual classification of terms based on references in the text corpus.,Manual classification text corpus


### Historical Analysis


In [10]:
import arxiv

def build_abs_query(query):
    terms = query.strip().split()
    if len(terms) == 1:
        return f"abs:{terms[0]}"
    # abs:word1 AND abs:word2 AND abs:word3
    return " AND ".join(f"abs:{term}" for term in terms)

def search_arxiv(query, end_year):
  client = arxiv.Client()

  query = build_abs_query(query)

  search = arxiv.Search(
    query = f'{query} AND submittedDate:[200001010000 TO {end_year}12310000]',
    sort_by = arxiv.SortCriterion.SubmittedDate,
    sort_order = arxiv.SortOrder.Ascending,
    max_results = 100
  )
  
  
  results = client.results(search)
  return list(results)

# result_arxiv = search_arxiv(methods_list[0], str(published_year))
# # methods_list[3]


In [76]:
from papers_retrieval import *
import time
import random


from dotenv import load_dotenv
load_dotenv(dotenv_path=".env")


search_semantic_scolar = getReferencePaper()

methods_list = [req.search_query for req in parsed.requirements if req.category == Category.method_technique]

all_results = []

for method_query in methods_list:
    # result_arxiv = search_arxiv(method_query, str(2019))
    result_semantic_scolar = search_semantic_scolar.query_search(method_query, str(published_year))

    for paper in result_semantic_scolar['data']:
        all_results.append({
            "method_query": method_query,
            "title": paper['title'],
            "paperId": paper['paperId'],
            "publicationDate": paper['publicationDate'],
            "citationCount": paper['citationCount'],
            "publicationType": paper['publicationTypes'][0] if paper['publicationTypes'] else None
        })
    time.sleep(random.randint(7,11))  # To respect API rate limits

# Convert to DataFrame
df_results = pd.DataFrame(all_results)

s2k-f3clylS9FPO3B882vUQGe5t4pQzcmxdTAhxQYmNC


KeyError: 'data'

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
df_results = pd.DataFrame(all_results)
df_results

In [ ]:
pd.DataFrame(all_results)

In [74]:
url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
# url = "https://api.semanticscholar.org/graph/v1/paper/search/"

query_params = {
    "query": "Neural network training Word2vec",
    "fields": "title,citationCount,publicationTypes,publicationDate",
    "year": f"-2024",
    "sort": "publicationDate:asc"
}
headers = {"x-api-key": "s2k-f3clylS9FPO3B882vUQGe5t4pQzcmxdTAhxQYmNC"}
response = requests.get(url, params=query_params, headers=headers).json()

In [ ]:
df_results = pd.DataFrame(response['data'])
df_results["publicationDate"] = pd.to_datetime(df_results["publicationDate"], errors="coerce")
df_results["year"] = df_results["publicationDate"].dt.year

# Count titles per method_query per year
yearly_counts = df_results.groupby(['method_query', 'year']).size().reset_index(name='count')

In [ ]:
df_results

In [486]:
yearly_counts

,method_query,year,count
0,Negation impact sentiment analysis,2018,1
1,Neural network training Word2vec,2014,1
2,Neural network training Word2vec,2016,1
3,Neural network training Word2vec,2017,5
4,Neural network training Word2vec,2018,9
5,Neural network training Word2vec,2019,15
6,Sentiment analysis finance texts,2013,1
7,Sentiment analysis finance texts,2018,1


In [457]:
# methods_list = [req.arxiv_search_query for req in parsed.requirements if req.category == Category.method_technique]

# published_year, methods_list, abstract
print(abstract)

Textual analysis is an increasingly important field in accounting and finance research, primarily focusing on the analysis of English texts. The thesis discusses several important adaptions and extensions as well as use cases for textual analysis of German-speaking finance-related texts. The research is divided into three interconnected studies.
The first study addresses the limitations of sentiment analysis for German financial texts by reforming and extending the first finance-specific German dictionary developed by Bannier et al. (2019). Through proposing several reforms and extensions of the existing dictionary and evaluating the most appropriate measurement of the tone of textual documents in finance, this study enhances the accuracy and reliability of sentiment analysis. 
The second study builds upon these advancements by further expanding the dictionary with 11,179 additional words, including basic forms and synonyms. This expanded dictionary is then used to analyze the sentimen

## Automate

In [153]:
import pandas as pd

df = pd.read_excel("/Users/ariq/Public/Data/thesis_feasibility/New Dataset_.xlsx", sheet_name="filtere")

In [196]:
len(row['Abstract'])

2262

In [ ]:
import os, json

abstracts_df = df[df["Academic Level"]=="grant"][["title","Abstract","Published Year","Academic Level"]]
output_file = "abstract_requirements.csv"
batch_size = 5  # Save every 5 abstracts

# Load already processed abstracts
if os.path.exists(output_file):
    processed = set(pd.read_csv(output_file)['source_title'])
    file_exists = True
else:
    processed = set()
    file_exists = False

results = []



for i, row in abstracts_df.iterrows():
    # Skip if already processed
    if row['title'] in processed:
        continue
    
    try:
        # Extract requirements via LLM
        if len(row['Abstract']) <10:
            continue
        resp = llm.invoke(prompt.format(abstract=row['Abstract']))
        parsed = AnalysisResult.model_validate(json.loads(resp.content))
        
        # Convert each requirement to a row
        for req in parsed.requirements:
            results.append({
                'category': str(req.category),
                'name': req.name,
                'description': req.description,
                'estimated_difficulty': str(req.estimated_difficulty),
                'search_query': req.search_query,
                'source_title': row['title'],
                'year': row['Published Year'],
                'academic_level': row['Academic Level'],
                'pred_academic_level': parsed.overall_suitable_degree,
                'status': 'success'
            })
        
        # Save batch every N abstracts
        if len(results) >= batch_size:
            pd.DataFrame(results).to_csv(output_file, mode='a' if file_exists else 'w', 
                                         header=not file_exists, index=False)
            processed.update({r['source_title'] for r in results})
            print(f"✓ Saved {len(results)} requirements")
            results = []
            file_exists = True
    
    except Exception as e:
        print(f"\n❌ FAILED at: {row['title']}")
        print(f"Error: {str(e)}")
        print(f"\nAlready processed: {len(processed)} abstracts")
        print(f"Saved to: {os.path.abspath(output_file)}")
        # break

# Save remaining results
if results:
    pd.DataFrame(results).to_csv(output_file, mode='a' if file_exists else 'w', 
                                 header=not file_exists, index=False)
    print(f"✓ Saved final {len(results)} requirements")

print(f"\n✓ Done! Results saved to: {os.path.abspath(output_file)}")

✓ Saved 5 requirements
✓ Saved 7 requirements
✓ Saved 6 requirements
✓ Saved 7 requirements

❌ FAILED at: EAGER: ATAROS: Automatic Tagging and Recognition of Stance
Error: 2 validation errors for AnalysisResult
requirements.6.search_query
  Field required [type=missing, input_value={'category': 'human_effor...difficulty': 'advanced'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
requirements.7.search_query
  Field required [type=missing, input_value={'category': 'compute', '...iculty': 'intermediate'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

Already processed: 481 abstracts
Saved to: /Users/ariq/Public/Data/thesis_feasibility/abstract_requirements.csv
✓ Saved 5 requirements
✓ Saved 6 requirements

❌ FAILED at: Workshop: Support for a workshop on scientific research applications of natural language technologies
Error: 2 validation errors for AnalysisResult
requirements.6.search_query


KeyboardInterrupt: 

In [189]:
import pandas as pd

sampled_df = pd.read_csv("abstract_requirements.csv")
sampled_df = sampled_df.drop_duplicates(subset=["source_title", "year"], keep="last")

frac = 0.3  # keep 30% of each year's rows
sampled = (
    sampled_df.groupby( "academic_level", group_keys=False)
      .apply(lambda g: g.sample(frac=frac, random_state=42))
)

/var/folders/wx/8d312jjx3_75qjml8rv9p6n00000gn/T/ipykernel_23372/969776771.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(frac=frac, random_state=42))


In [202]:
sampled_df = pd.read_excel("abstract_requirements 2.xlsx")

## Deduplication

Four-phase pipeline — no feedback loops:
1. **Embed** name+description with a sentence-transformer
2. **Cluster** via agglomerative clustering (cosine distance, tight threshold)
3. **Name** each cluster with one small LLM call (no merge/split decisions)
4. **Fuzzy dedup** canonical names with rapidfuzz (deterministic, no LLM)


In [ ]:
# Install dependencies (run once)
# !pip install sentence-transformers scikit-learn rapidfuzz


In [ ]:
import logging
import pandas as pd
from tools import get_model
from dotenv import load_dotenv
from deduplication import deduplicate_requirements, explore_threshold

load_dotenv(dotenv_path=".env")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

llm = get_model()


In [ ]:
sampled_df = pd.read_excel("abstract_requirements 2.xlsx")

method_df = (
    sampled_df[sampled_df["category"] == "Category.method_technique"]
    .copy()
    .reset_index(drop=True)
)

print(f"{len(method_df)} method_technique rows")
method_df[["name", "description"]].head()


In [ ]:
# Explore clustering thresholds on a 200-row sample before committing.
# Look for a threshold where avg_size is 2-5 and singleton_pct is below ~40%.
threshold_report = explore_threshold(
    method_df,
    name_col="name",
    desc_col="description",
)
threshold_report


In [ ]:
# Run the full pipeline.
# Adjust distance_threshold based on explore_threshold() output above.
deduped_df = deduplicate_requirements(
    method_df,
    llm,
    name_col="name",
    desc_col="description",
    distance_threshold=0.28,   # cosine distance; lower = tighter clusters
    fuzzy_threshold=88,        # rapidfuzz score; above = same canonical name
    delay=(1, 3),
)

print(f"Rows          : {len(deduped_df)}")
print(f"Canonical names: {deduped_df.canonical_name.nunique()}")
print(f"Clusters       : {deduped_df.cluster_id.nunique()}")
deduped_df[["name", "canonical_name", "cluster_id"]].head(20)


In [ ]:
# Inspect cluster contents — spot-check quality.
cluster_summary = (
    deduped_df.groupby(["cluster_id", "canonical_name"])["name"]
    .apply(lambda x: list(x.unique())[:6])
    .reset_index()
    .rename(columns={"name": "sample_raw_names"})
    .sort_values("cluster_id")
)
cluster_summary


In [ ]:
# Merge back onto the full dataframe and save.
full_deduped = sampled_df.copy().reset_index(drop=True)

method_mask = full_deduped["category"] == "Category.method_technique"
full_deduped.loc[method_mask, "canonical_name"] = deduped_df["canonical_name"].values
full_deduped.loc[method_mask, "cluster_id"] = deduped_df["cluster_id"].values

out_path = "abstract_requirements_deduped.xlsx"
full_deduped.to_excel(out_path, index=False)
print(f"Saved → {out_path}")


## Method Matching

Given an input method/technique description, find previously analyzed thesis 
abstracts that used the same or a similar method. This is retrieval, not 
clustering — no LLM involved. An embedding index is built once over the 
requirements corpus, then ranked by cosine similarity to the query text.


In [ ]:
import logging
import pandas as pd
from method_matcher import MethodMatcher

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

sampled_df = pd.read_excel("abstract_requirements 2.xlsx")
method_df = (
    sampled_df[sampled_df["category"] == "Category.method_technique"]
    .copy()
    .reset_index(drop=True)
)

matcher = MethodMatcher(method_df, name_col="name", desc_col="description")


In [ ]:
# Persist the index so repeated runs don't re-embed ~1,600 rows each time.
matcher.save_index("method_index.npz")

# To reload later without re-embedding:
# matcher = MethodMatcher.load_index("method_index.npz", method_df)


In [ ]:
# Single query example
query = "logistic regression for binary classification"

matches = matcher.search(
    query,
    top_k=10,
    min_similarity=0.45,
    extra_cols=["source_title", "year", "academic_level"],
)
matches


In [ ]:
# Batch query example — more efficient than calling search() in a loop
queries = [
    "sentiment analysis of financial text",
    "named entity recognition",
    "convolutional neural network for image classification",
]

batch_matches = matcher.search_many(
    queries,
    top_k=5,
    min_similarity=0.45,
    extra_cols=["source_title", "year"],
)
batch_matches


## Method Adoption Timeline

For an input abstract: extract its methods, look up when each method first 
appeared in the literature (Semantic Scholar, earliest paper), then plot:
- Two smooth lines: thesis count and grant count per year (base distribution)
- One translucent vertical line per method, at its origin year
- A dot on the relevant curve for every year a similar method was used by an
  existing thesis/grant in the corpus (via the embedding-based MethodMatcher)


In [ ]:
# Step 1: extract requirements from the input abstract (reuses the prompt/llm
# from the top of this notebook).
input_abstract = abstract  # swap in any abstract text

result = llm.invoke(prompt.format(abstract=input_abstract))
parsed_input = AnalysisResult.model_validate(json.loads(result.content))

input_methods = [
    {"name": req.name, "search_query": req.search_query}
    for req in parsed_input.requirements
    if req.category == Category.method_technique
]
input_methods


In [ ]:
# Step 2: look up the origin year for each method (earliest paper found).
# This hits the Semantic Scholar API once per method with a polite delay —
# expect ~10s per method.
from method_history import get_origin_years

queries = [m["search_query"] for m in input_methods]
origin_years = get_origin_years(queries, end_year=int(published_year))

for m in input_methods:
    m["origin_year"] = origin_years.get(m["search_query"])

input_methods


In [ ]:
# Step 3: reuse (or build) the MethodMatcher over the per-requirement corpus.
# Reuses matcher from the Method Matching section above if already built.
try:
    matcher
except NameError:
    from method_matcher import MethodMatcher
    sampled_df = pd.read_excel("abstract_requirements 2.xlsx")
    method_df = (
        sampled_df[sampled_df["category"] == "Category.method_technique"]
        .copy()
        .reset_index(drop=True)
    )
    matcher = MethodMatcher(method_df, name_col="name", desc_col="description")


In [ ]:
# Step 4: load the one-row-per-abstract dataframe used for the base distribution
# (different from method_df, which has one row per extracted requirement).
abstracts_df = pd.read_excel("/Users/ariq/Public/Data/thesis_feasibility/New Dataset.xlsx", sheet_name="filtered")


In [ ]:
# Install dependency for the interactive chart (run once)
# !pip install plotly


In [ ]:
# Step 5: plot (interactive).
# Hover over a line/dot for details, click a legend entry to toggle a
# method's vertical line + dots together, scroll/drag to zoom.
from timeline_plot import plot_method_adoption_timeline_plotly

fig = plot_method_adoption_timeline_plotly(
    abstracts_df,
    input_methods,
    matcher,
    year_col="Published Year",
    level_col="Academic Level",
    matched_year_col="year",
    matched_level_col="academic_level",
    matched_title_col="source_title",
    input_year=int(published_year),
    min_similarity=0.45,
    save_path="method_adoption_timeline.html",
)
fig.show()
